In [34]:
import pandas as pd

pd.set_option('display.max_columns', None)

df = pd.read_parquet("data/games_2024_2025_2026.parquet")

# todays_games = df[df["game_date"] == "2026-04-08"]

home_favored = df[
    (df["sp_kbb_diff"] > 3) &
    (df["sp_xfip_diff"] < -0.5) &
    (df["offense_diff"] > 10)
]

away_favored = df[
    (df["sp_kbb_diff"] < -3) &
    (df["sp_xfip_diff"] > 0.5) &
    (df["offense_diff"] < -10)
]

mismatches = pd.concat([home_favored, away_favored])
# mismatches["favored_team"] = mismatches.apply(
#     lambda row: row["home_team_name"] if row["sp_kbb_diff"] > 0 else row["away_team_name"],
#     axis=1
# )

mismatches["favored_home"] = mismatches["sp_kbb_diff"] > 0
mismatches["favored_won"] = (
    (mismatches["favored_home"] & (mismatches["home_win"] == 1)) |
    (~mismatches["favored_home"] & (mismatches["home_win"] == 0))
)

mismatches["game_date"] = pd.to_datetime(mismatches["game_date"])

mismatches["year"] = mismatches["game_date"].dt.year
mismatches["month"] = mismatches["game_date"].dt.month

mismatches.groupby(["year", "month"]).agg(
    games=("favored_won", "size"),
    win_rate=("favored_won", "mean")
).sort_index()

games  win_rate
year month                 
2024 3         23  0.608696
     4         26  0.576923
     5         22  0.818182
     6         28  0.642857
     7         14  0.785714
     8         27  0.851852
     9         20  0.850000
2025 3         16  0.812500
     4         10  0.700000
     5         22  0.863636
     6         28  0.678571
     7          6  0.666667
     8         27  0.592593
     9         12  0.833333
2026 3         28  0.642857
     4         21  0.476190